# ECE input diagnosis (`ece-input-diagnose-1.0`)

This notebook tests why `derived_8.4-ece-model-salvage-1.1`'s Best-1.1 prediction tracks rainfall while the daily ground truth looks flat (`ece_all_sensors_input_overlay_rainfall.png`). It reconstructs **hourly** sensor series from the raw burst CSVs (Seattle Time) on the 3 largest pooled rain days plus nonzero neighbors, and compares them against the daily mean that feeds the model. The goal is to see whether the sensor captures the event within the day and to quantify how daily averaging smooths or obscures that response. Sensor coverage here means burst-sample availability per hour, since the raw files carry no QC/state flag column.


In [1]:
from pathlib import Path
import sys

import matplotlib
import numpy as np
import pandas as pd
import yaml

_CAND = Path("experiment/ece-input-diagnose-1.0")
EXP_DIR = _CAND.resolve() if _CAND.exists() else Path.cwd().resolve()
assert (EXP_DIR / "config.yaml").exists(), f"config.yaml not under {EXP_DIR}"
PROJECT_ROOT = EXP_DIR.parents[2]
print("EXP_DIR:", EXP_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)

config = yaml.safe_load((EXP_DIR / "config.yaml").read_text())
print("experiment:", config["experiment"])

PRIOR = PROJECT_ROOT / config["repo_paths"]["prior_audit_dir"]
sys.path.insert(0, str(PRIOR))
from audit_coverage import load_ece_raw
from compare_averaging import daily_estimators

FIG_DIR = EXP_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("matplotlib:", matplotlib.__version__)

EXP_DIR: /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/ece-input-diagnose-1.0
PROJECT_ROOT: /scratch/group/p.cis250607.000/MDR-Project
experiment: ece-input-diagnose-1.0
matplotlib: 3.10.9


## Load daily drivers and Best-1.1 predictions

The next cell loads the daily ECE test split, the Best-1.1 mean prediction over seeds 42/7/13 from `predictions.csv`, and verifies the 3-largest-day selection (pooled daily max: 2026-07-23 at 10.7 mm, 2026-07-26 at 6.8 mm, 2026-08-02 at 2.3 mm) with the nonzero neighbor (only 2026-07-25), extended by dry context days (day-before for event A, day-after for events B and C), giving 7 focus days in 3 event windows.


In [2]:
TARGET = "soil_moisture_5cm"
BEST_ID = config["params"]["best_model_id"]
SEEDS = [int(s) for s in config["params"]["seeds"]]
FOCUS_DAYS = list(config["params"]["focus_days"])
WINDOWS = {k: list(v) for k, v in config["params"]["event_windows"].items()}
STATIONS = list(config["ece_files"].keys())

daily = pd.read_csv(
    PROJECT_ROOT / config["repo_paths"]["ece_test"],
    usecols=["station_id", "date", TARGET, "precip_mm", "LST_modis", "G_API",
             "G_DSLR", "G_rain_sum_3d", "G_rain_sum_7d"],
    low_memory=False,
)
daily["date"] = pd.to_datetime(daily["date"])
assert len(daily) == 150 and daily["station_id"].nunique() == 5, daily.shape

pooled = daily.groupby("date")["precip_mm"].max().sort_values(ascending=False)
print("pooled daily-max ranking:")
print(pooled.head(8).to_string())
top3 = list(pooled.head(3).index.strftime("%Y-%m-%d"))
assert top3 == ["2026-07-23", "2026-07-26", "2026-08-02"], top3
assert FOCUS_DAYS == ["2026-07-22", "2026-07-23", "2026-07-25", "2026-07-26",
                      "2026-07-27", "2026-08-02", "2026-08-03"], FOCUS_DAYS
assert WINDOWS == {"event_A": ["2026-07-22", "2026-07-23"],
                   "event_B": ["2026-07-25", "2026-07-26", "2026-07-27"],
                   "event_C": ["2026-08-02", "2026-08-03"]}, WINDOWS
for day, before, after in [("2026-07-23", "2026-07-22", "2026-07-24"),
                           ("2026-07-26", "2026-07-25", "2026-07-27"),
                           ("2026-08-02", "2026-08-01", "2026-08-03")]:
    b = pooled.get(pd.Timestamp(before), 0.0)
    a = pooled.get(pd.Timestamp(after), 0.0)
    print(f"{day}: before {before} max={b:.1f}, after {after} max={a:.1f}")
for day in ["2026-07-22", "2026-07-27", "2026-08-03"]:
    assert pooled.get(pd.Timestamp(day), 0.0) == 0.0, f"context day {day} is not dry!"
    assert day in set(daily["date"].dt.strftime("%Y-%m-%d")), f"context day {day} missing rows!"
print("context days 07-22/07-27/08-03: dry (0.0 mm) with full rows.")

pred = pd.read_csv(PROJECT_ROOT / config["repo_paths"]["salvage_predictions"],
                   usecols=["model_id", "seed", "dataset", "window", "station_id",
                            "date", "target", "prediction"], low_memory=False)
sel = pred[pred["model_id"].eq(BEST_ID) & pred["seed"].isin(SEEDS)
           & pred["dataset"].eq("ece_spatial")].copy()
assert len(sel) == 5 * 30 * len(SEEDS), len(sel)
best_mean = sel.groupby(["station_id", "date"], as_index=False)["prediction"].mean()
best_mean = best_mean.rename(columns={"prediction": "best_1_1"})
best_mean["date"] = pd.to_datetime(best_mean["date"])
truth = sel.groupby(["station_id", "date"], as_index=False)["target"].mean()
truth = truth.rename(columns={"target": "target_predsrc"})
truth["date"] = pd.to_datetime(truth["date"])
merged = daily.merge(best_mean, on=["station_id", "date"], validate="one_to_one")
merged = merged.merge(truth, on=["station_id", "date"], validate="one_to_one")
assert np.allclose(merged[TARGET], merged["target_predsrc"]), "target mismatch!"
focus = merged[merged["date"].isin(pd.to_datetime(FOCUS_DAYS))].copy()
assert len(focus) == 5 * len(FOCUS_DAYS), len(focus)
print(focus[["station_id", "date", TARGET, "best_1_1", "precip_mm",
             "G_rain_sum_3d"]].sort_values(["date", "station_id"]).to_string(index=False))


pooled daily-max ranking:
date
2026-07-23    10.7
2026-07-26     6.8
2026-08-02     2.3
2026-07-25     0.8
2026-08-15     0.5
2026-07-28     0.3
2026-08-12     0.2
2026-07-22     0.0
2026-07-23: before 2026-07-22 max=0.0, after 2026-07-24 max=0.0
2026-07-26: before 2026-07-25 max=0.8, after 2026-07-27 max=0.0
2026-08-02: before 2026-08-01 max=0.0, after 2026-08-03 max=0.0
context days 07-22/07-27/08-03: dry (0.0 mm) with full rows.


             station_id       date  soil_moisture_5cm  best_1_1  precip_mm  G_rain_sum_3d
    ECE_BBG_Lost_Meadow 2026-07-22           0.050300  0.073203        0.0            0.0
        ECE_BBG_Main_St 2026-07-22           0.056064  0.080686        0.0            0.0
ECE_Renton_Garden_North 2026-07-22           0.197635  0.069179        0.0            0.0
 ECE_Renton_Garden_Shed 2026-07-22           0.072107  0.069172        0.0            0.0
        ECE_Renton_Home 2026-07-22           0.017618  0.071156        0.0            0.0
    ECE_BBG_Lost_Meadow 2026-07-23           0.059340  0.076211        6.1            6.1
        ECE_BBG_Main_St 2026-07-23           0.061891  0.082427        6.1            6.1
ECE_Renton_Garden_North 2026-07-23           0.188672  0.079805       10.7           10.7
 ECE_Renton_Garden_Shed 2026-07-23           0.078603  0.079529       10.7           10.7
        ECE_Renton_Home 2026-07-23           0.018937  0.079942       10.7           10.7
    ECE_BB

## Reconstruct hourly sensor series and coverage

The next cell parses each raw burst file with the shared `load_ece_raw` helper (Seattle Time preferred, same as `ece_pipe.py`), bins samples into hourly means, and tabulates per-station-day coverage (sample count, distinct hours). It also reuses `daily_estimators` for simple-mean versus hourly-weighted-mean versus median on the focus days. Coverage gaps are sensor-data gaps: missing hours are kept as gaps and never interpolated.

In [3]:
RAW_DIR = PROJECT_ROOT / config["repo_paths"]["ece_raw_dir"]
MIN_HOURS = int(config["params"]["min_hours_per_day"])

raw_by_station = {}
for station, fname in config["ece_files"].items():
    fpath = RAW_DIR / fname
    assert fpath.exists(), f"missing raw file: {fpath}"
    df = load_ece_raw(fpath)
    df["station_id"] = station
    raw_by_station[station] = df
print("raw spans:")
for station in STATIONS:
    df = raw_by_station[station]
    print(f"  {station}: n={len(df)} {df['date'].min().date()}..{df['date'].max().date()}")

FOCUS_TS = pd.to_datetime(FOCUS_DAYS)
hourly_rows = []
for station in STATIONS:
    df = raw_by_station[station]
    sub = df[df["date"].isin(FOCUS_TS)].copy()
    sub["sm_frac"] = sub["sm_pct"] / 100.0
    g = sub.groupby([sub["date"], sub["hour"]])["sm_frac"].agg(["mean", "count"]).reset_index()
    g.columns = ["date", "hour", "hourly_mean", "hourly_n"]
    g["station_id"] = station
    g["timestamp"] = pd.to_datetime(g["date"]) + pd.to_timedelta(g["hour"], unit="h")
    hourly_rows.append(g)
hourly = pd.concat(hourly_rows, ignore_index=True).sort_values(
    ["station_id", "timestamp"]).reset_index(drop=True)
print(f"hourly rows on focus days: {len(hourly)}")

cov_rows = []
for station in STATIONS:
    df = raw_by_station[station]
    for day in FOCUS_TS:
        sub = df[df["date"] == day]
        cov_rows.append({
            "station_id": station,
            "date": day,
            "n_samples": int(len(sub)),
            "n_hours": int(sub["hour"].nunique()) if len(sub) else 0,
            "meets_18h": bool(sub["hour"].nunique() >= MIN_HOURS) if len(sub) else False,
        })
coverage = pd.DataFrame(cov_rows).sort_values(["date", "station_id"]).reset_index(drop=True)
print(coverage.to_string(index=False))
assert coverage["meets_18h"].all(), "a focus day fails the 18/24h minimum-hour rule!"
low = coverage[coverage["n_hours"] < 24]
print(f"min distinct hours on focus days: {int(coverage['n_hours'].min())}; "
      f"days below full 24h: {len(low)} (shown as line breaks, never interpolated)")

est_rows = []
for station in STATIONS:
    est = daily_estimators(raw_by_station[station])
    est["station_id"] = station
    est_rows.append(est[est["date"].isin(FOCUS_TS)])
estimators = pd.concat(est_rows, ignore_index=True)
print(estimators[["station_id", "date", "n", "n_hours", "simple_mean",
                  "hourly_weighted_mean", "median", "diurnal_range"]]
      .sort_values(["date", "station_id"]).to_string(index=False))

hourly.to_csv(EXP_DIR / "hourly_comparison.csv", index=False)
coverage.to_csv(EXP_DIR / "focus_coverage.csv", index=False)
print("saved: hourly_comparison.csv, focus_coverage.csv")


raw spans:
  ECE_BBG_Main_St: n=13600 2026-07-19..2026-08-20
  ECE_BBG_Lost_Meadow: n=12427 2026-07-19..2026-08-20
  ECE_Renton_Home: n=13126 2026-07-19..2026-08-20
  ECE_Renton_Garden_North: n=12311 2026-07-19..2026-08-20
  ECE_Renton_Garden_Shed: n=9132 2026-07-19..2026-08-20
hourly rows on focus days: 836
             station_id       date  n_samples  n_hours  meets_18h
    ECE_BBG_Lost_Meadow 2026-07-22        465       24       True
        ECE_BBG_Main_St 2026-07-22        408       24       True
ECE_Renton_Garden_North 2026-07-22        445       24       True
 ECE_Renton_Garden_Shed 2026-07-22        235       22       True
        ECE_Renton_Home 2026-07-22        436       24       True
    ECE_BBG_Lost_Meadow 2026-07-23        456       24       True
        ECE_BBG_Main_St 2026-07-23        422       24       True
ECE_Renton_Garden_North 2026-07-23        403       24       True
 ECE_Renton_Garden_Shed 2026-07-23        242       24       True
        ECE_Renton_Home 2026-0

             station_id       date   n  n_hours  simple_mean  hourly_weighted_mean  median  diurnal_range
    ECE_BBG_Lost_Meadow 2026-07-22 465       24     0.050300              0.049046 0.05730       0.037055
        ECE_BBG_Main_St 2026-07-22 408       24     0.056064              0.056160 0.05410       0.046379
ECE_Renton_Garden_North 2026-07-22 445       24     0.197635              0.197115 0.20010       0.067168
 ECE_Renton_Garden_Shed 2026-07-22 235       22     0.072107              0.072966 0.07250       0.050988
        ECE_Renton_Home 2026-07-22 436       24     0.017618              0.017446 0.01860       0.027206
    ECE_BBG_Lost_Meadow 2026-07-23 456       24     0.059340              0.058394 0.06280       0.026409
        ECE_BBG_Main_St 2026-07-23 422       24     0.061891              0.061911 0.06060       0.016236
ECE_Renton_Garden_North 2026-07-23 403       24     0.188672              0.189245 0.19250       0.053580
 ECE_Renton_Garden_Shed 2026-07-23 242       2

## Hourly line charts per event window

The next cell draws one figure per event window (A: 2026-07-22/23; B: 2026-07-25/26/27; C: 2026-08-02/03), faceted by station. Each panel shows the hourly sensor mean line over the window, the daily-mean segment actually fed to the model (black dashed), the hourly-weighted-mean segment (green dotted), the Best-1.1 prediction and daily target markers, and the daily `precip_mm` label. Missing hours appear as line breaks, with sample-count coverage noted per day. Shared y-limits keep spike sizes comparable across stations.


In [4]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

saved_figs = []
for window_name, day_strs in WINDOWS.items():
    days = pd.to_datetime(day_strs)
    t0 = days.min()
    t1 = days.max() + pd.Timedelta(hours=23)
    full_idx = pd.date_range(t0, t1, freq="h")
    fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
    for ax, station in zip(axes, STATIONS):
        h = hourly[hourly["station_id"].eq(station)].set_index("timestamp")
        h = h.reindex(full_idx)
        ax.plot(h.index, h["hourly_mean"], color="tab:blue", linewidth=1.6,
                label="Hourly sensor mean")
        for day in days:
            dstr = day.strftime("%Y-%m-%d")
            est = estimators[(estimators["station_id"].eq(station))
                             & (estimators["date"] == day)]
            mrow = merged[(merged["station_id"].eq(station)) & (merged["date"] == day)]
            assert len(est) == 1 and len(mrow) == 1, (station, dstr)
            est = est.iloc[0]
            mrow = mrow.iloc[0]
            d0 = day
            d1 = day + pd.Timedelta(hours=23)
            ax.hlines(est["simple_mean"], d0, d1, colors="black",
                      linestyles="--", linewidth=1.6, label="Daily mean (model input)")
            ax.hlines(est["hourly_weighted_mean"], d0, d1, colors="tab:green",
                      linestyles=":", linewidth=1.4, label="Hourly-weighted mean")
            noon = day + pd.Timedelta(hours=12)
            ax.plot(noon, mrow["best_1_1"], marker="x", color="tab:orange",
                    markersize=8, linestyle="None", label="Best-1.1 prediction")
            ax.plot(noon, mrow[TARGET], marker="D", color="black",
                    markersize=5, linestyle="None", label="Daily target")
            ax.text(day + pd.Timedelta(hours=1), 0.92,
                    f"{dstr[5:]} P={mrow['precip_mm']:.1f}mm n={int(est['n'])}h={int(est['n_hours'])}",
                    fontsize=8, va="top", ha="left", transform=ax.get_xaxis_transform(),
                    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))
        ax.set_title(station)
        ax.grid(alpha=0.25)
        ax.set_ylabel("Soil moisture (frac)")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
    fig.autofmt_xdate(rotation=20)
    handles, labels = axes[0].get_legend_handles_labels()
    seen, uh, ul = set(), [], []
    for hh, ll in zip(handles, labels):
        if ll not in seen:
            seen.add(ll)
            uh.append(hh)
            ul.append(ll)
    fig.suptitle(f"ECE hourly sensor vs daily mean — {window_name} "
                 f"({', '.join(day_strs)})", y=0.99)
    fig.tight_layout(rect=[0, 0.04, 1, 0.94])
    fig.legend(uh, ul, loc="lower center", bbox_to_anchor=(0.5, 0.005),
               ncol=5, fontsize=9)
    out = FIG_DIR / f"hourly_{window_name}_{'_'.join(d.strftime('%Y%m%d') for d in days)}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_figs.append(out)
    print("saved:", out.name, f"({out.stat().st_size/1e6:.2f} MB)")
print(f"figures: {len(saved_figs)}")


saved: hourly_event_A_20260722_20260723.png (0.29 MB)


saved: hourly_event_B_20260725_20260726_20260727.png (0.30 MB)


saved: hourly_event_C_20260802_20260803.png (0.29 MB)
figures: 3


## Smoothing summary: does the sensor see the rain?

The next cell quantifies, per focus station-day, the within-day sensor swing (hourly max, min, range, spike above the daily mean) against the day-to-day move of the daily mean and of the Best-1.1 prediction. If hourly range dwarfs the daily step while the daily mean barely moves across the rain event, daily averaging is smoothing out a response the sensor actually recorded. Results are saved to `smoothing_summary.csv` for the README.

In [5]:
srows = []
for station in STATIONS:
    for day in FOCUS_TS:
        h = hourly[(hourly["station_id"].eq(station)) & (hourly["date"] == day)]
        est = estimators[(estimators["station_id"].eq(station))
                         & (estimators["date"] == day)].iloc[0]
        mrow = merged[(merged["station_id"].eq(station)) & (merged["date"] == day)].iloc[0]
        prev = merged[(merged["station_id"].eq(station))
                      & (merged["date"] == day - pd.Timedelta(days=1))]
        prev_target = float(prev[TARGET].iloc[0]) if len(prev) else np.nan
        prev_pred = float(prev["best_1_1"].iloc[0]) if len(prev) else np.nan
        hmax = float(h["hourly_mean"].max())
        hmin = float(h["hourly_mean"].min())
        srows.append({
            "station_id": station,
            "date": day.strftime("%Y-%m-%d"),
            "precip_mm": float(mrow["precip_mm"]),
            "daily_mean": float(est["simple_mean"]),
            "hourly_weighted": float(est["hourly_weighted_mean"]),
            "simple_minus_hourly": float(est["simple_mean"] - est["hourly_weighted_mean"]),
            "hourly_max": hmax,
            "hourly_min": hmin,
            "hourly_range": hmax - hmin,
            "spike_above_daily": hmax - float(est["simple_mean"]),
            "dip_below_daily": float(est["simple_mean"]) - hmin,
            "d_target_vs_prev_day": float(mrow[TARGET]) - prev_target,
            "d_pred_vs_prev_day": float(mrow["best_1_1"]) - prev_pred,
            "target": float(mrow[TARGET]),
            "best_1_1": float(mrow["best_1_1"]),
            "n_hours": int(est["n_hours"]),
        })
smoothing = pd.DataFrame(srows).sort_values(["date", "station_id"]).reset_index(drop=True)
smoothing.to_csv(EXP_DIR / "smoothing_summary.csv", index=False)
print(smoothing[["station_id", "date", "precip_mm", "daily_mean", "hourly_range",
                 "spike_above_daily", "d_target_vs_prev_day",
                 "d_pred_vs_prev_day"]].to_string(index=False))
print("\nmedian hourly_range on focus days: %.4f" % smoothing["hourly_range"].median())
print("median |d_target_vs_prev_day|: %.4f"
      % smoothing["d_target_vs_prev_day"].abs().median())
print("median |d_pred_vs_prev_day|: %.4f"
      % smoothing["d_pred_vs_prev_day"].abs().median())
print("max |simple_minus_hourly|: %.4f" % smoothing["simple_minus_hourly"].abs().max())
print("saved: smoothing_summary.csv")
rainy = smoothing[smoothing["precip_mm"] > 0]
print("rainy station-days: %d of %d" % (len(rainy), len(smoothing)))
print("median hourly_range on rainy days: %.4f" % rainy["hourly_range"].median())
print("median |d_target_vs_prev_day| on rainy days: %.4f"
      % rainy["d_target_vs_prev_day"].abs().median())

             station_id       date  precip_mm  daily_mean  hourly_range  spike_above_daily  d_target_vs_prev_day  d_pred_vs_prev_day
    ECE_BBG_Lost_Meadow 2026-07-22        0.0    0.050300      0.037055           0.015955             -0.000259            0.000138
        ECE_BBG_Main_St 2026-07-22        0.0    0.056064      0.046379           0.027019             -0.001398            0.000581
ECE_Renton_Garden_North 2026-07-22        0.0    0.197635      0.067168           0.037465              0.002664            0.000253
 ECE_Renton_Garden_Shed 2026-07-22        0.0    0.072107      0.050988           0.026410             -0.007333            0.000029
        ECE_Renton_Home 2026-07-22        0.0    0.017618      0.027206           0.009588             -0.002493           -0.000552
    ECE_BBG_Lost_Meadow 2026-07-23        6.1    0.059340      0.026409           0.008386              0.009040            0.003008
        ECE_BBG_Main_St 2026-07-23        6.1    0.061891      0.0162

## Hourly rainfall reference

The next cell loads `hourly_rain_reference.csv` (Open-Meteo archive `rain,precipitation`, fetched by `fetch_hourly_rain.py` with the same `timezone: auto` parameters as `weather_pipe.py`). It proves the hourly series is the identical source as the model's daily `precip_mm` by reproducing the daily sums, and tabulates per focus station-day the rain total, peak hour, and number of rainy hours. Timezone note: `auto` resolves to America/Los_Angeles, which is PDT (UTC-7) for the whole July-August window with no DST transition inside it, so the naive timestamps compare directly against the Seattle-wall-hour sensor bins.

In [6]:
rain = pd.read_csv(EXP_DIR / "hourly_rain_reference.csv")
rain["timestamp"] = pd.to_datetime(rain["timestamp"])
rain["date"] = rain["timestamp"].dt.floor("D")
rain["hour"] = rain["timestamp"].dt.hour
assert len(rain) == 5 * 33 * 24, f"unexpected hourly rain row count: {len(rain)}"
fcheck = rain[rain["date"].isin(FOCUS_TS)].groupby("station_id")["hour"].nunique()
assert (fcheck == 24).all(), fcheck.to_string()

rain_daily = rain.groupby(["station_id", "date"], as_index=False)["precip_mm"].sum()
chk = rain_daily.merge(daily[["station_id", "date", "precip_mm"]],
                       on=["station_id", "date"],
                       suffixes=("_hoursum", "_test"), validate="one_to_one")
maxdiff = (chk["precip_mm_hoursum"] - chk["precip_mm_test"]).abs().max()
print(f"max |hourly-sum - test.csv precip_mm| = {maxdiff:.2e}")
assert maxdiff < 1e-9, "hourly rain is not the same source as the daily model input!"

rrows = []
for station in STATIONS:
    for day in FOCUS_TS:
        sub = rain[(rain["station_id"].eq(station)) & (rain["date"] == day)]
        tot = float(sub["precip_mm"].sum())
        pk = sub.loc[sub["precip_mm"].idxmax()]
        rrows.append({
            "station_id": station,
            "date": day.strftime("%Y-%m-%d"),
            "rain_total_mm": round(tot, 2),
            "rain_peak_hour": int(pk["hour"]),
            "rain_peak_mm": round(float(pk["precip_mm"]), 2),
            "n_rain_hours": int((sub["precip_mm"] > 0).sum()),
        })
raintab = pd.DataFrame(rrows).sort_values(["date", "station_id"]).reset_index(drop=True)
print(raintab.to_string(index=False))

max |hourly-sum - test.csv precip_mm| = 5.55e-17
             station_id       date  rain_total_mm  rain_peak_hour  rain_peak_mm  n_rain_hours
    ECE_BBG_Lost_Meadow 2026-07-22            0.0               0           0.0             0
        ECE_BBG_Main_St 2026-07-22            0.0               0           0.0             0
ECE_Renton_Garden_North 2026-07-22            0.0               0           0.0             0
 ECE_Renton_Garden_Shed 2026-07-22            0.0               0           0.0             0
        ECE_Renton_Home 2026-07-22            0.0               0           0.0             0
    ECE_BBG_Lost_Meadow 2026-07-23            6.1               0           3.8             7
        ECE_BBG_Main_St 2026-07-23            6.1               0           3.8             7
ECE_Renton_Garden_North 2026-07-23           10.7               0           6.7             6
 ECE_Renton_Garden_Shed 2026-07-23           10.7               0           6.7             6
        ECE

## Hourly rainfall overlay figures

The next cell redraws one figure per event window (A: 2026-07-22/23; B: 2026-07-25/26/27; C: 2026-08-02/03) with hourly Open-Meteo precipitation bars on a twin rain axis behind the hourly sensor line. This shows when within the day the rain fell relative to the sensor's midday dry-down dips — the timing information the daily `precip_mm` bar in the salvage overlay cannot carry. Missing sensor hours still appear as line breaks; rain bars are complete (33-day continuous fetch).


In [7]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

saved_rain_figs = []
for window_name, day_strs in WINDOWS.items():
    days = pd.to_datetime(day_strs)
    t0 = days.min()
    t1 = days.max() + pd.Timedelta(hours=23)
    full_idx = pd.date_range(t0, t1, freq="h")
    wmax = float(rain[(rain["station_id"].isin(STATIONS))
                      & (rain["timestamp"] >= t0)
                      & (rain["timestamp"] <= t1)]["precip_mm"].max())
    rain_ylim = (0.0, max(wmax * 1.25, 0.5))
    fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
    first_twin = None
    for ax, station in zip(axes, STATIONS):
        h = hourly[hourly["station_id"].eq(station)].set_index("timestamp")
        h = h.reindex(full_idx)
        ax.plot(h.index, h["hourly_mean"], color="tab:blue", linewidth=1.6,
                label="Hourly sensor mean")
        for day in days:
            est = estimators[(estimators["station_id"].eq(station))
                             & (estimators["date"] == day)].iloc[0]
            mrow = merged[(merged["station_id"].eq(station))
                          & (merged["date"] == day)].iloc[0]
            ax.hlines(est["simple_mean"], day, day + pd.Timedelta(hours=23),
                      colors="black", linestyles="--", linewidth=1.6,
                      label="Daily mean (model input)")
            noon = day + pd.Timedelta(hours=12)
            ax.plot(noon, mrow[TARGET], marker="D", color="black",
                    markersize=5, linestyle="None", label="Daily target")
            ax.plot(noon, mrow["best_1_1"], marker="x", color="tab:orange",
                    markersize=8, linestyle="None", label="Best-1.1 prediction")
        rr = rain[(rain["station_id"].eq(station))
                  & (rain["timestamp"] >= t0) & (rain["timestamp"] <= t1)]
        ax2 = ax.twinx()
        if first_twin is None:
            first_twin = ax2
        ax2.bar(rr["timestamp"], rr["precip_mm"], color="gray", alpha=0.45,
                width=0.035, label="Hourly rain (mm)")
        ax2.set_ylim(*rain_ylim)
        ax2.set_ylabel("Rain (mm/h)")
        ax.set_title(station)
        ax.grid(alpha=0.25)
        ax.set_ylabel("Soil moisture (frac)")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
    fig.autofmt_xdate(rotation=20)
    handles, labels = axes[0].get_legend_handles_labels()
    h2, l2 = first_twin.get_legend_handles_labels()
    seen, uh, ul = set(), [], []
    for hh, ll in list(zip(handles, labels)) + list(zip(h2, l2)):
        if ll not in seen:
            seen.add(ll)
            uh.append(hh)
            ul.append(ll)
    fig.suptitle(f"ECE hourly sensor with hourly rainfall — {window_name} "
                 f"({', '.join(day_strs)})", y=0.99)
    fig.tight_layout(rect=[0, 0.04, 1, 0.94])
    fig.legend(uh, ul, loc="lower center", bbox_to_anchor=(0.5, 0.005),
               ncol=6, fontsize=9)
    out = FIG_DIR / f"hourly_rain_{window_name}_{'_'.join(d.strftime('%Y%m%d') for d in days)}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_rain_figs.append(out)
    print("saved:", out.name, f"({out.stat().st_size/1e6:.2f} MB)")
print(f"rain figures: {len(saved_rain_figs)}")

saved: hourly_rain_event_A_20260722_20260723.png (0.27 MB)


saved: hourly_rain_event_B_20260725_20260726_20260727.png (0.30 MB)


saved: hourly_rain_event_C_20260802_20260803.png (0.28 MB)
rain figures: 3


## Rain-to-sensor lag analysis

The next cell quantifies timing: per focus station-day, the hour of peak rain versus the hours of the sensor's daily maximum and minimum (from the hourly means), plus the lag from rain peak to sensor minimum. It also pools all focus station-hours to correlate the hour-to-hour sensor change against rain at 0–6 h lags, testing how fast the sensor responds. Results go to `rain_lag_summary.csv`.

In [8]:
lrows = []
for station in STATIONS:
    for day in FOCUS_TS:
        h = hourly[(hourly["station_id"].eq(station)) & (hourly["date"] == day)]
        rr = rain[(rain["station_id"].eq(station)) & (rain["date"] == day)]
        tot = float(rr["precip_mm"].sum())
        peak_h = int(rr.loc[rr["precip_mm"].idxmax(), "hour"]) if tot > 0 else None
        max_h = int(h.loc[h["hourly_mean"].idxmax(), "hour"])
        min_h = int(h.loc[h["hourly_mean"].idxmin(), "hour"])
        lrows.append({
            "station_id": station,
            "date": day.strftime("%Y-%m-%d"),
            "rain_total_mm": round(tot, 2),
            "rain_peak_hour": peak_h,
            "sensor_max_hour": max_h,
            "sensor_min_hour": min_h,
            "min_lag_after_peak_h": (min_h - peak_h) % 24 if peak_h is not None else None,
        })
lagtab = pd.DataFrame(lrows).sort_values(["date", "station_id"]).reset_index(drop=True)
lagtab.to_csv(EXP_DIR / "rain_lag_summary.csv", index=False)
print(lagtab.to_string(index=False))

pool = hourly[hourly["date"].isin(FOCUS_TS)].merge(
    rain[["station_id", "timestamp", "precip_mm"]].rename(columns={"precip_mm": "rain_mmh"}),
    on=["station_id", "timestamp"], validate="one_to_one")
pool = pool.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
pool["d_sensor"] = pool.groupby("station_id")["hourly_mean"].diff()
day_start = pool["hour"] == 0
print("\npooled corr(d_sensor[h], rain[h-lag]) over focus station-hours:")
for lag in range(7):
    sub = pool[~pool["d_sensor"].isna()].copy()
    sub["rain_lag"] = sub.groupby("station_id")["rain_mmh"].shift(lag)
    sub = sub[~sub["rain_lag"].isna()]
    r_lag = float(sub[["d_sensor", "rain_lag"]].corr().iloc[0, 1])
    print(f"  lag {lag}h: r={r_lag:+.3f} (n={len(sub)})")
print("saved: rain_lag_summary.csv")

             station_id       date  rain_total_mm  rain_peak_hour  sensor_max_hour  sensor_min_hour  min_lag_after_peak_h
    ECE_BBG_Lost_Meadow 2026-07-22            0.0             NaN               22               13                   NaN
        ECE_BBG_Main_St 2026-07-22            0.0             NaN               21               13                   NaN
ECE_Renton_Garden_North 2026-07-22            0.0             NaN                4               15                   NaN
 ECE_Renton_Garden_Shed 2026-07-22            0.0             NaN                5               15                   NaN
        ECE_Renton_Home 2026-07-22            0.0             NaN                5               14                   NaN
    ECE_BBG_Lost_Meadow 2026-07-23            6.1             0.0                4               14                  14.0
        ECE_BBG_Main_St 2026-07-23            6.1             0.0               23                0                   0.0
ECE_Renton_Garden_North 

## Completion

All tables above are sourced from executed cells, and the linked figures were generated during this notebook execution. The hourly series confirm the sensors record large within-day swings on both rain and dry days (median hourly range 0.0349 over all 7 focus days, 0.0287 on rainy days — roughly 10–14x the median day-to-day target step of 0.0025), while the daily mean fed to the model sits nearly flat through each event. The dry context days (07-22, 07-27, 08-03) replicate the same midday-dry-down timing with no rain at all, proving the cycle is diurnal rather than rain-driven; the hourly rain overlay shows event rain falling mostly overnight, which the 24 h mean then averages against the dry afternoon. Coverage on the focus days is 22–24 distinct hours everywhere, so the flat daily line is not a missing-data artifact; the 18/24 h minimum-hour rule changes nothing here. Simple-mean versus hourly-weighted-mean differs by at most 0.0031, so the averaging definition is second-order next to the smoothing effect itself.


In [9]:
print("Notebook complete — all report sections above are generated from experiment artifacts.")
print(f"figures: {[p.name for p in sorted(FIG_DIR.glob('hourly_*.png'))]}")

Notebook complete — all report sections above are generated from experiment artifacts.
figures: ['hourly_event_A_20260722_20260723.png', 'hourly_event_B_20260725_20260726_20260727.png', 'hourly_event_C_20260802_20260803.png', 'hourly_rain_event_A_20260722_20260723.png', 'hourly_rain_event_B_20260725_20260726_20260727.png', 'hourly_rain_event_C_20260802_20260803.png']
